# Predicting Student Health Risk — v4: CatBoost (saves OOF + test probabilities, GPU-enabled)
### Kaggle Playground Series S6E7

Same as your working v4 (CV = **0.94902**), now with:
- `task_type="GPU"` enabled — remember to switch the Accelerator to **GPU P100** (or T4) in
  Notebook Options, and re-run from the top after switching (changing accelerator restarts the kernel)
- Saves `catboost_oof_proba.npy` (out-of-fold probabilities on train) and `catboost_test_proba.npy`
  (averaged probabilities on test) — needed for the v5 ensemble notebook

Run via **Save Version → Save & Run All (Commit)**.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import balanced_accuracy_score
from catboost import CatBoostClassifier
import warnings
warnings.filterwarnings("ignore")

DATA_DIR = "/kaggle/input/competitions/playground-series-s6e7"


## 1. Load the data

In [2]:
train = pd.read_csv(f"{DATA_DIR}/train.csv")
test = pd.read_csv(f"{DATA_DIR}/test.csv")
sample_sub = pd.read_csv(f"{DATA_DIR}/sample_submission.csv")

target_col = "health_condition"
feature_cols = [c for c in train.columns if c not in [target_col, "id"]]
print("Train shape:", train.shape, " Test shape:", test.shape)

Train shape: (690088, 15)  Test shape: (295753, 14)


## 2. Preprocessing

In [3]:
X = train[feature_cols].copy()
y = train[target_col].copy()
X_test = test[feature_cols].copy()

cat_cols = X.select_dtypes(include="object").columns.tolist()
num_cols = X.select_dtypes(exclude="object").columns.tolist()

for c in num_cols:
    X[c] = X[c].fillna(X[c].median())
    X_test[c] = X_test[c].fillna(X[c].median())

for c in cat_cols:
    X[c] = X[c].fillna("missing").astype(str)
    X_test[c] = X_test[c].fillna("missing").astype(str)

cat_feature_indices = [X.columns.get_loc(c) for c in cat_cols]

target_encoder = LabelEncoder()
y_enc = target_encoder.fit_transform(y)
print(dict(zip(target_encoder.classes_, range(len(target_encoder.classes_)))))

{'at-risk': 0, 'fit': 1, 'unhealthy': 2}


## 3. Train and validate with CatBoost (GPU) — now also collecting OOF probabilities

In [4]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
fold_scores = []
test_preds_folds = []
oof_proba = np.zeros((len(X), 3))

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y_enc)):
    X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_val = y_enc[train_idx], y_enc[val_idx]

    model = CatBoostClassifier(
        iterations=2000,
        learning_rate=0.05,
        depth=8,
        loss_function="MultiClass",
        auto_class_weights="Balanced",
        cat_features=cat_feature_indices,
        random_seed=42,
        verbose=False,
        early_stopping_rounds=50,
        task_type="GPU",
        devices="0",
    )
    model.fit(X_tr, y_tr, eval_set=(X_val, y_val))

    val_pred_proba = model.predict_proba(X_val)
    oof_proba[val_idx] = val_pred_proba

    val_pred = np.argmax(val_pred_proba, axis=1)
    score = balanced_accuracy_score(y_val, val_pred)
    fold_scores.append(score)
    print(f"Fold {fold+1} balanced accuracy: {score:.5f}")

    test_preds_folds.append(model.predict_proba(X_test))

print(f"\nMean CV balanced accuracy: {np.mean(fold_scores):.5f} (+/- {np.std(fold_scores):.5f})")

oof_score = balanced_accuracy_score(y_enc, np.argmax(oof_proba, axis=1))
print(f"OOF balanced accuracy (whole train set): {oof_score:.5f}")

Fold 1 balanced accuracy: 0.94969
Fold 2 balanced accuracy: 0.95086
Fold 3 balanced accuracy: 0.94868
Fold 4 balanced accuracy: 0.94902
Fold 5 balanced accuracy: 0.94698

Mean CV balanced accuracy: 0.94905 (+/- 0.00127)
OOF balanced accuracy (whole train set): 0.94905


## 4. Save submission + OOF + test probabilities

In [5]:
avg_test_proba = np.mean(test_preds_folds, axis=0)
final_preds_enc = np.argmax(avg_test_proba, axis=1)
final_preds = target_encoder.inverse_transform(final_preds_enc)

submission = pd.DataFrame({"id": test["id"], "health_condition": final_preds})
assert list(submission.columns) == list(sample_sub.columns)
assert len(submission) == len(sample_sub)
submission.to_csv("submission.csv", index=False)

np.save("catboost_oof_proba.npy", oof_proba)
np.save("catboost_test_proba.npy", avg_test_proba)
print("Saved catboost_oof_proba.npy, catboost_test_proba.npy, submission.csv")
print("Class order:", target_encoder.classes_)
submission.head()

Saved catboost_oof_proba.npy, catboost_test_proba.npy, submission.csv
Class order: ['at-risk' 'fit' 'unhealthy']


,id,health_condition
0,690088,unhealthy
1,690089,unhealthy
2,690090,at-risk
3,690091,at-risk
4,690092,unhealthy


## Next step
Download `catboost_oof_proba.npy` and `catboost_test_proba.npy` from this notebook's Output tab.
Combine with the LightGBM v3 OOF/test files and `y_true_enc.npy` in the v5 ensemble notebook.